In [1]:
# Benchmark notebook setup
import os
import sys
import subprocess

REPO_DIR = "/content/OpenEnv-WolfeClick"
GITHUB_REPO = "Atharva2099/OpenEnv-WolfeClick"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", f"https://github.com/{GITHUB_REPO}.git", REPO_DIR],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

    %pip uninstall -y unsloth vllm torchaudio torchvision xformers || true
    %pip install -q "torch==2.10.0" \
        "transformers==4.57.1" \
        "trl==0.24.0" \
        "peft==0.18.1" \
        "accelerate>=1.0.0" \
        "bitsandbytes" \
        "huggingface_hub"
    %pip install -q "poke-env==0.8.3" pydantic numpy
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)

for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("Benchmark environment ready.")


Benchmark environment ready.


In [2]:
# Debug Showdown startup with visible logs
import subprocess
import time
import os
import socket

SHOWDOWN_DIR = "/content/pokemon-showdown"

if not os.path.exists(SHOWDOWN_DIR):
    subprocess.run(
        ["git", "clone", "--depth=1", "https://github.com/smogon/pokemon-showdown.git", SHOWDOWN_DIR],
        check=True,
    )

subprocess.run(["pkill", "-f", "pokemon-showdown"], capture_output=True)
time.sleep(1)

showdown_proc = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    cwd=SHOWDOWN_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(5)

# Print startup logs
if showdown_proc.poll() is not None:
    print("Showdown exited early. Logs:")
    print(showdown_proc.stdout.read())
    raise RuntimeError("Pokemon Showdown failed to start")

s = socket.socket()
result = s.connect_ex(("127.0.0.1", 8000))
s.close()

if result != 0:
    print("Port 8000 still closed. Recent logs:")
    print(showdown_proc.stdout.read())
    raise RuntimeError("Pokemon Showdown server not reachable on port 8000")

print(f"Pokemon Showdown server started (PID {showdown_proc.pid})")


Pokemon Showdown server started (PID 48605)


In [ ]:
# Benchmark two checkpoints on the same env budget
import json
import os
import re
import time
import statistics
from dataclasses import dataclass

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from smogon_rl.config import EnvConfig
from smogon_rl.openenv_sync_env import PokemonShowdownEnv
from smogon_rl.action_space import (
    enumerate_actions,
    build_action_instructions,
    extract_action_json_from_text,
    parse_llm_action,
)

HF_TOKEN = os.environ.get("HF_TOKEN", "<YOUR_HF_TOKEN>")

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
SYSTEM_PROMPT = (
    "You are a competitive Pokemon battler. "
    "Analyse the battle state below and choose exactly one action. "
    "You MUST output ONLY a single JSON object — no explanation, no markdown fences, just the raw JSON.\n"
    '{"action": "move" | "switch", "choice": "Exact Name of Move or Pokemon"}'
)

MAX_NEW_TOKENS = 24
TEMPERATURE = 0.3
NUM_GENERATIONS = 4
BATTLES = 2

@dataclass
class CheckpointSpec:
    label: str
    repo_id: str
    revision: str

CHECKPOINTS = [
    CheckpointSpec(
        label="GRPO Run 1",
        repo_id="Atharva2099/openenv-smogon-rl",
        revision="grpo-qwen3-4b-run2",
    ),
    CheckpointSpec(
        label="GRPO Run 2",
        repo_id="Atharva2099/openenv-smogon-rl",
        revision="grpo-qwen3-4b-run3",
    ),
]

def build_prompt_messages(state_str, valid_actions):
    instructions = build_action_instructions(valid_actions)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{state_str}\n\n{instructions}"},
    ]

@torch.no_grad()
def generate_action_candidates(model, tokenizer, state_str, valid_actions):
    messages = build_prompt_messages(state_str, valid_actions)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        chat_template_kwargs={"enable_thinking": False},
    )
    device = model.get_input_embeddings().weight.device
    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_return_sequences=NUM_GENERATIONS,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=0.95,
        top_k=20,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]

def load_checkpoint(spec):
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        use_fast=True,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(
        model,
        spec.repo_id,
        revision=spec.revision,
        token=HF_TOKEN,
        is_trainable=False,
        autocast_adapter_dtype=False,
    )
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True
    return model, tokenizer

def benchmark_checkpoint(spec):
    model, tokenizer = load_checkpoint(spec)
    env = PokemonShowdownEnv(
        config=EnvConfig(
            battle_format="gen4randombattle",
            verbose_logging=True,
            log_every_n_steps=10,
            poll_heartbeat_seconds=5.0,
        )
    )

    total_steps = 0
    total_reward = 0.0
    model_invalid = 0
    env_illegal = 0
    format_hits = 0
    battle_rewards = []
    battle_lengths = []
    wins = 0
    losses = 0
    unknown = 0

    t0 = time.perf_counter()

    for battle_idx in range(BATTLES):
        print(f"[{spec.label}] Battle {battle_idx + 1}/{BATTLES} started", flush=True)
        state_str = env.reset()
        done = False
        battle_reward = 0.0
        battle_len = 0

        while not done:
            battle = env._ensure_battle()
            valid_actions = enumerate_actions(battle)
            if not valid_actions:
                break

            candidates = generate_action_candidates(model, tokenizer, state_str, valid_actions)
            chosen_str = None

            for c in candidates:
                clean = re.sub(r"<think>.*?</think>", "", c, flags=re.DOTALL).strip()
                try:
                    parse_llm_action(clean, valid_actions)
                    chosen_str = clean
                    format_hits += 1
                    break
                except ValueError:
                    pass

                extracted = extract_action_json_from_text(c)
                if extracted is not None:
                    try:
                        parse_llm_action(extracted, valid_actions)
                        chosen_str = extracted
                        format_hits += 1
                        break
                    except ValueError:
                        pass

            if chosen_str is None:
                model_invalid += 1
                fb = valid_actions[0]
                chosen_str = json.dumps({"action": fb.action_type, "choice": fb.choice})

            state_str, reward, done, info = env.step(chosen_str)
            total_steps += 1
            battle_len += 1
            total_reward += float(reward)
            battle_reward += float(reward)

            if info.get("action_illegal", False):
                env_illegal += 1

        finished_battle = env._ensure_battle()
        won = getattr(finished_battle, "won", None)
        lost = getattr(finished_battle, "lost", None)
        if won is True:
            wins += 1
        elif lost is True:
            losses += 1
        else:
            unknown += 1

        battle_rewards.append(battle_reward)
        battle_lengths.append(battle_len)
        print(
            f"[{spec.label}] Battle {battle_idx + 1}/{BATTLES} finished | "
            f"steps={battle_len} reward={battle_reward:.3f}",
            flush=True,
        )

    dt = time.perf_counter() - t0
    return {
        "Checkpoint": spec.label,
        "Battles": BATTLES,
        "Wins": wins,
        "Losses": losses,
        "Unknown": unknown,
        "Format Hit Rate": f"{100.0 * format_hits / max(1, total_steps):.1f}%",
        "Model Invalid": model_invalid,
        "Env Illegal": env_illegal,
        "Avg Reward / Turn": round(total_reward / max(1, total_steps), 3),
        "Avg Battle Reward": round(statistics.mean(battle_rewards) if battle_rewards else 0.0, 3),
        "Avg Battle Length": round(statistics.mean(battle_lengths) if battle_lengths else 0.0, 1),
        "Wall Time (s)": round(dt, 1),
    }

results = [benchmark_checkpoint(spec) for spec in CHECKPOINTS]

headers = list(results[0].keys())
print("\n# Latest Benchmark Results\n")
print("| " + " | ".join(headers) + " |")
print("| " + " | ".join(["---"] * len(headers)) + " |")
for row in results:
    print("| " + " | ".join(str(row[h]) for h in headers) + " |")


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[GRPO Run 1] Battle 1/2 started
[PokeEnvClient] Background event loop started.
[PokeEnvClient] Players created and connected.
[PokeEnvClient] Launching new battle in format gen4randombattle.
[PokeEnvClient] Battle update received: turn=1, finished=False.
[PokemonShowdownEnv] Battle 1 started at turn=1 (format=gen4randombattle).
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=1, previous_turn=1, elapsed=10.0s.
[PokeEnvClient] Battle update received: turn=2, finished=False.
[PokemonShowdownEnv] battle=1 step=1 turn=2 reward=4.778 running_reward=4.778 illegal_action=False done=False
[PokeEnvClient] Submitted action to RLPlayer queue.
[PokeEnvClient] Waiting for turn advance: current_turn=2, previous_turn=2, elapsed=5.0s.
[PokeEnvClient] Waiting for turn advance: current_turn=2, previous_turn=2, elapsed=10.0s.
[PokeEnvClient] Waiting for turn a

In [ ]:
# Watch one live battle in the browser
import webbrowser

WATCH_SPEC = CheckpointSpec(
    label="Watch Run 2",
    repo_id="Atharva2099/openenv-smogon-rl",
    revision="grpo-qwen3-4b-run2",
)
OPEN_BROWSER = True

model, tokenizer = load_checkpoint(WATCH_SPEC)
env = PokemonShowdownEnv(
    config=EnvConfig(
        battle_format="gen4randombattle",
        verbose_logging=True,
        log_every_n_steps=1,
        poll_heartbeat_seconds=5.0,
    )
)

state_str = env.reset()
battle = env._ensure_battle()
room_url = f"http://127.0.0.1:8000/{battle.battle_tag}"
print(f"Watching room: {room_url}")
if OPEN_BROWSER:
    webbrowser.open(room_url)

done = False
total_reward = 0.0
step_idx = 0

while not done:
    step_idx += 1
    battle = env._ensure_battle()
    valid_actions = enumerate_actions(battle)
    if not valid_actions:
        print("No valid actions available; ending battle.")
        break

    candidates = generate_action_candidates(model, tokenizer, state_str, valid_actions)
    chosen_str = None

    for c in candidates:
        clean = re.sub(r"<think>.*?</think>", "", c, flags=re.DOTALL).strip()
        try:
            parse_llm_action(clean, valid_actions)
            chosen_str = clean
            break
        except ValueError:
            pass

        extracted = extract_action_json_from_text(c)
        if extracted is not None:
            try:
                parse_llm_action(extracted, valid_actions)
                chosen_str = extracted
                break
            except ValueError:
                pass

    if chosen_str is None:
        fb = valid_actions[0]
        chosen_str = json.dumps({"action": fb.action_type, "choice": fb.choice})

    state_str, reward, done, info = env.step(chosen_str)
    total_reward += float(reward)
    print(
        f"step={step_idx} turn={info.get('turn')} action={chosen_str} "
        f"reward={reward:.3f} total_reward={total_reward:.3f} done={done}",
        flush=True,
    )

finished_battle = env._ensure_battle()
print(
    f"Battle complete | won={getattr(finished_battle, 'won', None)} "
    f"lost={getattr(finished_battle, 'lost', None)} total_reward={total_reward:.3f}"
)
print(f"Room URL: {room_url}")
